In [1]:
import pandas as pd
import openpyxl

def main():
    find_coordinate2(r'D:\Renstra\20260406 - KK Highlevel Real Mar 2026.xlsx','LR-SEMEN', "REAL")
    
    df = get_excel_data(r'D:\Renstra\20260406 - KK Highlevel Real Mar 2026.xlsx', 'LR-SEMEN', "B, T, AI", 2, 120, 2)
    df_rkap = get_rkap(df)
    
    print(df_rkap)
                       
def get_excel_data(address, sheet_name=0, usecols=None, skiprows=0, nrows=None, header=None):
    df = pd.read_excel(
        address,
        sheet_name=sheet_name,
        usecols=usecols,
        skiprows=skiprows,
        nrows=nrows,
        thousands='.', 
        decimal=','
    )
    return df

def get_rkap(df):
    # --- CLEANING PROCESS ---
    df.columns = ['Account','RKAPJan2026', 'REALFeb2026']
    # 2. Remove the first row if it contains the redundant sub-header (Jan, Feb, Mar)
    df = df.iloc[1:].reset_index(drop=True)
    # 3. Clean numeric strings
    # DROP NaN specifically from the 'Account' column
    # This removes any row where 'Account' is empty
    df.dropna(subset=['Account'], inplace=True)
    #df = df[df['Account'].notna()]
    #df = df[df.columns.dropna()]
    #df = df.dropna()
    # Excel often exports numbers with dots as thousands separators or parentheses for negatives.
    def clean_currency(value):
        if pd.isna(value) or value == '-':
            return 0.0
        val_str = str(value).replace('.', '')   # Remove thousands separator
        if '(' in val_str:                      # Handle negatives like (133.620)
            val_str = '-' + val_str.replace('(', '').replace(')', '')
        return pd.to_numeric(val_str, errors='coerce')

    numeric_cols = ['RKAPJan2026', 'REALFeb2026']
    for col in numeric_cols:
        df[col] = df[col].apply(clean_currency)

    # 3. Handle cases where the cell isn't NaN but is just whitespace or a dash
    # This is common in "unstructured" Excel files
    df = df[df['Account'].astype(str).str.strip() != '']
    df = df[df['Account'].astype(str).str.strip() != '-']
    # 4. Drop completely empty rows
    df = df.dropna(how='all')
    return df.reset_index(drop=True)

def normalize_to_time_series(df):
    # 1. Melt the months into a single column
    # This turns [Account, Jan, Feb, Mar] into [Account, Month, Value]
    df_melted = df.melt(
        id_vars=['Account'], 
        var_name='Month_Year', 
        value_name='Amount'
    )

    # 2. Convert 'RKAP_Jan_2026' strings into real Python Dates
    # This is crucial for Looker Studio timelines
    month_map = {
        'RKAPJan2026': '2026-01-01'
    }
    df_melted['Date'] = pd.to_datetime(df_melted['Month_Year'].map(month_map))

    # 3. Pivot the 'Account' labels into Columns
    # This creates columns for Revenue, Beban Pokok, etc., for each Date
    df_normalized = df_melted.pivot_table(
        index='Date', 
        columns='Account', 
        values='Amount', 
        aggfunc='sum'
    ).reset_index()

    return df_normalized

def find_coordinate(address, sheet, target_value):
    df = get_excel_data(address, sheet, header=None)

    # Define the value you are looking for
    target_value = target_value

    # Find the coordinates where the value exists
    # .stack() turns the grid into a list of (row, col) pairs
    match = df[df == target_value].stack().index
    print(match)
    if not match.empty:
        first_occurrence = match[0] # Returns (row_index, column_index)
        print(f"Found at Row: {first_occurrence[0]}, Column: {first_occurrence[1]}")
    else:
        print("Value not found.")
        
def find_coordinate2(address, sheet, target_value):
    wb = openpyxl.load_workbook(address , data_only=True)

    # 2. Access the sheet by name
    # Replace 'YourSheetName' with the actual name of the tab in Excel
    sheet = wb[sheet]
    found_cell = None

    for row in sheet.iter_rows():
        for cell in row:
            if cell.value == target_value:
                found_cell = cell
                break
        if found_cell:
            break

    if found_cell:
        print(f"Found at {found_cell.coordinate} (Row {found_cell.row}, Col {found_cell.column})")
    else:
        print("Value not found.")

def get_last_date(date):
    # Let's say you have a specific date
    current_date = pd.to_datetime('2026-02-15')

    # Move it to the end of the current month
    last_date_of_month = current_date + pd.offsets.MonthEnd(0)

    print(last_date_of_month) # Output: 2026-02-28
    return last_date_of_month

if __name__ == '__main__':
    main()

C:\Users\mochamad.ilmawan\Anaconda3\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


Found at D2 (Row 2, Col 4)
                           Account   RKAPJan2026   REALFeb2026
0   Gross Revenue (Cement+Clinker)  2.599090e+16  2.791646e+16
1             Gross Revenue Mortar  1.253504e+15  6.041350e+15
2                    Gross Revenue  2.611625e+16  2.797688e+16
3                    Ongkos Angkut -1.984622e+16 -2.110856e+16
4                          Revenue  2.413163e+15  2.586602e+16
..                             ...           ...           ...
91       Urusan Umum & Adm. Kantor  1.138502e+16  1.238608e+16
92        Beban Keuangan & Lainnya  6.204248e+15  9.069521e+15
93              Pajak dan Asuransi  7.175404e+15  6.643851e+13
94                         Promosi  2.825335e+15  3.036117e+16
95                Total Fixed Cost  9.744638e+15  9.861097e+15

[96 rows x 3 columns]
